## Task 2: Context Size vs Accuracy and Latency

**Hypothesis:** increasing the number of training rows that TabPFN sees will improve predictive accuracy at first, but the gains will flatten as context size grows. Train and prediction latency should increase with context size because the model has more context to condition on.

The experiment below varies only the number of sampled training rows. Each context size is evaluated across multiple random seeds, then summarized with mean values and standard-error error bars.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

In [ ]:
try:
    from tabpfn import TabPFNClassifier, TabPFNRegressor
except ImportError:
    raise ImportError(
        "Warning: Could not import TabPFN. Please run installation above and restart the session afterwards (Runtime > Restart Session)."
    )

# Data Science & Visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import time

# Other ML Models
from catboost import CatBoostClassifier, CatBoostRegressor

# Notebook UI/Display
from IPython.display import Markdown, display
from rich.console import Console
from rich.panel import Panel
from rich.rule import Rule
from sklearn.compose import make_column_selector, make_column_transformer

# Scikit-Learn: Data & Preprocessing
from sklearn.datasets import fetch_openml, load_breast_cancer

# Scikit-Learn: Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss, accuracy_score
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier, XGBRegressor

# This transformer will be used to handle categorical features for the baseline models
column_transformer = make_column_transformer(
    (
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        make_column_selector(dtype_include=["object", "category"]),
    ),
    remainder="passthrough",
)

In [ ]:
console = Console()

console.print(Panel.fit("[bold magenta]TabPFN Demo: Backend Selection[/bold magenta]"))
console.print("\nThis script can run TabPFN using one of two backends:")
console.print("  [bold]local:[/bold] Uses a local GPU (NVIDIA). Requires CUDA.")
console.print(
    "  [bold]client:[/bold] Uses the TabPFN API. Requires an internet connection and a free account."
)

backend = None
while backend is None:
  console.print(
      "\n[bold]Choose your backend[/bold]: - If no text box is shown, restart the cell.",
  )
  user_input = input("Enter 'local' or 'client' and press return:")
  if user_input not in ["local", "client"]:
    continue
  backend = user_input

console.print(
    f"\n✅ You have selected the '[bold green]{backend}[/bold green]' backend."
)

console.print(Rule(f"[bold]Setting up [cyan]{backend}[/cyan] backend[/bold]"))

if backend == "local":
    console.print("Attempting local backend setup...")
    import torch

    if not torch.cuda.is_available():
        console.print(
            "[bold red]Error:[/bold red] GPU device not found. For fast training, please enable GPU.",
            style="red",
        )
        console.print(
            "In Colab: Go to [bold]Runtime -> Change runtime type -> Hardware accelerator -> GPU.[/bold]",
            style="yellow",
        )
        raise SystemError("GPU device not found.")
    console.print("[bold green]✅ GPU is available.[/bold green]")

    # --- Prior Labs Authentication ---
    console.print(Rule("[bold]Prior Labs Authentication[/bold]"))
    console.print(
        "\nTabPFN model weights require a free [bold]Prior Labs account[/bold] and "
        "acceptance of the non-commercial license.\n"
    )

    import os
    import getpass

    tabpfn_token = None

    # 1. Try Colab secret TABPFN_TOKEN
    try:
        from google.colab import userdata
        tabpfn_token = userdata.get("TABPFN_TOKEN")
        if tabpfn_token:
            os.environ["TABPFN_TOKEN"] = tabpfn_token
            console.print("[bold green]✅ Found TABPFN_TOKEN in Colab secrets.[/bold green]")
    except Exception:
        pass

    # 2. If no token found, prompt the user
    if not tabpfn_token:
        console.print(
            Panel(
                "To get your access token:\n\n"
                "  1. Go to [link=https://ux.priorlabs.ai]ux.priorlabs.ai[/link] and sign up / log in\n"
                "  2. Accept the license at [link=https://ux.priorlabs.ai/account/licenses]ux.priorlabs.ai/account/licenses[/link]\n"
                "  3. Copy your Access Token from [link=https://ux.priorlabs.ai/account]ux.priorlabs.ai/account[/link]\n\n"
                "[bold yellow]Tip:[/bold yellow] Save the token as a Colab secret named "
                "[bold cyan]TABPFN_TOKEN[/bold cyan] to skip this step next time.",
                title="[bold]🔑 Prior Labs Access Token required",
                border_style="blue",
            )
        )
        while not tabpfn_token:
            token_input = getpass.getpass("Paste your TABPFN_TOKEN and press Enter: ")
            if token_input.strip():
                tabpfn_token = token_input.strip()
                os.environ["TABPFN_TOKEN"] = tabpfn_token
            else:
                console.print("[red]Token cannot be empty. Please try again.[/red]")

    console.print("")
    console.print("Importing local TabPFN library...")

    from tabpfn import TabPFNClassifier, TabPFNRegressor

    console.print("[bold green]✅ TabPFN (local) imported successfully.[/bold green]")
elif backend == "client":
    console.print("Attempting client backend setup...")
    console.print("Importing TabPFN client library...")
    from tabpfn_client import TabPFNClassifier, TabPFNRegressor, init

    init()
    console.print("[bold green]✅ TabPFN (client) initialized.[/bold green]")

In [ ]:
def run_context_size_experiment(X, y, context_sizes, seeds, test_size=0.20):
    """Randomly sample different training-context sizes and measure quality plus latency."""
    rows = []
    y = np.asarray(y)

    for seed in seeds:
        X_train_pool, X_test, y_train_pool, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=seed,
            stratify=y,
        )
        y_train_pool = np.asarray(y_train_pool)
        y_test = np.asarray(y_test)
        rng = np.random.default_rng(seed)

        for context_size in context_sizes:
            if context_size > len(X_train_pool):
                raise ValueError(
                    f"context_size={context_size} is larger than the available "
                    f"training pool of {len(X_train_pool)} rows"
                )

            sample_idx = rng.choice(len(X_train_pool), size=context_size, replace=False)
            X_train_context = X_train_pool.iloc[sample_idx]
            y_train_context = y_train_pool[sample_idx]

            model = TabPFNClassifier(random_state=seed)

            train_start = time.perf_counter()
            model.fit(X_train_context, y_train_context)
            train_end = time.perf_counter()

            predict_start = time.perf_counter()
            y_pred_proba = np.asarray(model.predict_proba(X_test), dtype=float)
            predict_end = time.perf_counter()

            y_pred_proba = np.clip(y_pred_proba, 1e-15, 1.0)
            y_pred_proba = y_pred_proba / y_pred_proba.sum(axis=1, keepdims=True)
            y_pred = model.classes_[np.argmax(y_pred_proba, axis=1)]

            train_latency = train_end - train_start
            predict_latency = predict_end - predict_start

            rows.append(
                {
                    "Seed": seed,
                    "Training Rows": context_size,
                    "Accuracy": accuracy_score(y_test, y_pred),
                    "ROC AUC": roc_auc_score(y_test, y_pred_proba[:, 1]),
                    "Log Loss": log_loss(y_test, y_pred_proba),
                    "Train Latency (s)": train_latency,
                    "Predict Latency (s)": predict_latency,
                    "End-to-End Latency (s)": train_latency + predict_latency,
                    "Train Latency Per Row (s)": train_latency / context_size,
                    "Predict Latency Per Test Row (s)": predict_latency / len(X_test),
                }
            )

    return pd.DataFrame(rows)

In [ ]:
pics_dir = "pics"
os.makedirs(pics_dir, exist_ok=True)

mean_plot_specs = [
    {
        "title": "Accuracy Mean",
        "mean_col": "Accuracy_mean",
        "sem_col": "Accuracy_sem",
        "ylabel": "Accuracy",
        "filename": "accuracy_mean.png",
    },
    {
        "title": "ROC AUC Mean",
        "mean_col": "ROC_AUC_mean",
        "sem_col": "ROC_AUC_sem",
        "ylabel": "ROC AUC",
        "filename": "roc_auc_mean.png",
    },
    {
        "title": "Log Loss Mean",
        "mean_col": "Log_Loss_mean",
        "sem_col": "Log_Loss_sem",
        "ylabel": "Log Loss",
        "filename": "log_loss_mean.png",
    },
    {
        "title": "Train Latency Mean",
        "mean_col": "Train_Latency_mean",
        "sem_col": "Train_Latency_sem",
        "ylabel": "Seconds",
        "filename": "train_latency_mean.png",
    },
    {
        "title": "Predict Latency Mean",
        "mean_col": "Predict_Latency_mean",
        "sem_col": "Predict_Latency_sem",
        "ylabel": "Seconds",
        "filename": "predict_latency_mean.png",
    },
    {
        "title": "End-to-End Latency Mean",
        "mean_col": "End_to_End_Latency_mean",
        "sem_col": "End_to_End_Latency_sem",
        "ylabel": "Seconds",
        "filename": "end_to_end_latency_mean.png",
    },
]

saved_plot_paths = []

for spec in mean_plot_specs:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        context_summary["Training Rows"],
        context_summary[spec["mean_col"]],
        yerr=context_summary[spec["sem_col"]],
        marker="o",
        linewidth=2,
        capsize=4,
    )
    ax.set_title(spec["title"])
    ax.set_xlabel("Training Rows Sampled")
    ax.set_ylabel(spec["ylabel"])
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    plot_path = os.path.join(pics_dir, spec["filename"])
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    saved_plot_paths.append(plot_path)
    plt.show()

saved_plot_paths


## Task 2: Robustness to Messy Data

**Hypothesis:** TabPFN will degrade more gracefully than baseline models as missing values and numeric outliers increase, while requiring less preprocessing when categorical columns are kept raw. Baselines usually need explicit imputation and encoding pipelines before they can handle the same messy inputs.

In [ ]:
def decode_raw_categoricals(X):
    """Decode byte-string categoricals from ARFF files into normal Python strings."""
    X_decoded = X.copy()
    categorical_cols = X_decoded.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:
        X_decoded[col] = X_decoded[col].map(
            lambda value: value.decode("utf-8") if isinstance(value, bytes) else value
        )

    return X_decoded


def make_messy_tabular_data(
    X_raw,
    missing_rate=0.0,
    outlier_rate=0.0,
    outlier_scale=10.0,
    raw_categoricals=True,
    seed=42,
):
    """Create a controlled messy-data variant by varying missingness, outliers, and categoricals.

    Parameters
    ----------
    X_raw : pd.DataFrame
        Original feature table before categorical encoding.
    missing_rate : float
        Fraction of cells to replace with np.nan. Applied across all columns.
    outlier_rate : float
        Fraction of numeric cells to perturb with large outliers.
    outlier_scale : float
        Multiplier for each numeric column standard deviation when creating outliers.
    raw_categoricals : bool
        If True, categorical columns stay as strings. If False, they are ordinal encoded.
    seed : int
        Random seed for reproducible corruption.
    """
    if not 0 <= missing_rate <= 1:
        raise ValueError("missing_rate must be between 0 and 1")
    if not 0 <= outlier_rate <= 1:
        raise ValueError("outlier_rate must be between 0 and 1")

    rng = np.random.default_rng(seed)
    X_messy = decode_raw_categoricals(X_raw).copy()

    categorical_cols = X_messy.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_cols = X_messy.select_dtypes(include=[np.number]).columns.tolist()

    # Add missing values across all feature columns.
    if missing_rate > 0:
        missing_mask = rng.random(X_messy.shape) < missing_rate
        X_messy = X_messy.mask(missing_mask)

    # Add large numeric outliers without changing categorical columns.
    if outlier_rate > 0 and numeric_cols:
        for col in numeric_cols:
            col_values = X_messy[col].astype(float)
            observed = col_values.notna().to_numpy()
            outlier_mask = (rng.random(len(X_messy)) < outlier_rate) & observed
            if not outlier_mask.any():
                continue

            col_std = col_values.std(skipna=True)
            if pd.isna(col_std) or col_std == 0:
                col_std = 1.0

            signs = rng.choice([-1, 1], size=outlier_mask.sum())
            X_messy.loc[outlier_mask, col] = (
                col_values.loc[outlier_mask] + signs * outlier_scale * col_std
            )

    # Keep categoricals raw for TabPFN-style testing, or encode for numeric-only baselines.
    if not raw_categoricals and categorical_cols:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_messy[categorical_cols] = encoder.fit_transform(X_messy[categorical_cols])

    metadata = {
        "missing_rate": missing_rate,
        "outlier_rate": outlier_rate,
        "outlier_scale": outlier_scale,
        "raw_categoricals": raw_categoricals,
        "seed": seed,
        "n_numeric_columns": len(numeric_cols),
        "n_categorical_columns": len(categorical_cols),
    }

    return X_messy, metadata


In [ ]:
# Missing-values-first experiment setup.
# These rates correspond to 0%, 5%, 10%, 15%, 20%, 25%, and 30% missing cells.
missing_rates = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

missing_value_variants = {}
missing_value_metadata = []

for missing_rate in missing_rates:
    X_missing, metadata = make_messy_tabular_data(
        X,
        missing_rate=missing_rate,
        outlier_rate=0.0,
        outlier_scale=10.0,
        raw_categoricals=True,
        seed=42,
    )

    label = f"{int(missing_rate * 100)}% missing"
    missing_value_variants[label] = X_missing
    missing_value_metadata.append(
        {
            **metadata,
            "label": label,
            "missing_cells": int(X_missing.isna().sum().sum()),
            "total_cells": int(X_missing.size),
            "observed_missing_rate": X_missing.isna().sum().sum() / X_missing.size,
        }
    )

missing_value_summary = pd.DataFrame(missing_value_metadata)[
    [
        "label",
        "missing_rate",
        "observed_missing_rate",
        "missing_cells",
        "total_cells",
        "raw_categoricals",
        "outlier_rate",
        "seed",
    ]
]

display(missing_value_summary)
display(missing_value_variants["10% missing"].head())


In [ ]:
def run_missing_value_tabpfn_experiment(X_raw, y, missing_rates, seeds, test_size=0.20):
    """Evaluate TabPFN as missing values increase while keeping raw categoricals."""
    rows = []
    y = np.asarray(y)

    for seed in seeds:
        for missing_rate in missing_rates:
            X_missing, metadata = make_messy_tabular_data(
                X_raw,
                missing_rate=missing_rate,
                outlier_rate=0.0,
                raw_categoricals=True,
                seed=seed,
            )

            X_train, X_test, y_train, y_test = train_test_split(
                X_missing,
                y,
                test_size=test_size,
                random_state=seed,
                stratify=y,
            )

            model = TabPFNClassifier(random_state=seed)
            model.fit(X_train, y_train)
            y_pred_proba = np.asarray(model.predict_proba(X_test), dtype=float)

            # Normalize to avoid metric warnings if backend probabilities have small numeric drift.
            y_pred_proba = np.clip(y_pred_proba, 1e-15, 1.0)
            y_pred_proba = y_pred_proba / y_pred_proba.sum(axis=1, keepdims=True)

            rows.append(
                {
                    "Seed": seed,
                    "Missing Rate": missing_rate,
                    "Missing Percent": int(missing_rate * 100),
                    "Observed Missing Rate": metadata["missing_rate"],
                    "ROC AUC": roc_auc_score(y_test, y_pred_proba[:, 1]),
                    "Log Loss": log_loss(y_test, y_pred_proba),
                }
            )

    return pd.DataFrame(rows)


missing_eval_seeds = [0, 1, 2]
missing_tabpfn_results = run_missing_value_tabpfn_experiment(
    X,
    y,
    missing_rates,
    missing_eval_seeds,
)

missing_tabpfn_summary = (
    missing_tabpfn_results
    .groupby(["Missing Rate", "Missing Percent"])
    .agg(
        ROC_AUC_mean=("ROC AUC", "mean"),
        ROC_AUC_sem=("ROC AUC", "sem"),
        Log_Loss_mean=("Log Loss", "mean"),
        Log_Loss_sem=("Log Loss", "sem"),
    )
    .reset_index()
    .fillna(0)
)

display(missing_tabpfn_summary)


In [ ]:
os.makedirs("pics", exist_ok=True)

missing_plot_specs = [
    {
        "title": "TabPFN ROC AUC vs Missing Values",
        "mean_col": "ROC_AUC_mean",
        "sem_col": "ROC_AUC_sem",
        "ylabel": "ROC AUC",
        "filename": "missing_values_tabpfn_roc_auc.png",
    },
    {
        "title": "TabPFN Log Loss vs Missing Values",
        "mean_col": "Log_Loss_mean",
        "sem_col": "Log_Loss_sem",
        "ylabel": "Log Loss",
        "filename": "missing_values_tabpfn_log_loss.png",
    },
]

missing_plot_paths = []

for spec in missing_plot_specs:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        missing_tabpfn_summary["Missing Percent"],
        missing_tabpfn_summary[spec["mean_col"]],
        yerr=missing_tabpfn_summary[spec["sem_col"]],
        marker="o",
        linewidth=2,
        capsize=4,
    )
    ax.set_title(spec["title"])
    ax.set_xlabel("Missing Values (%)")
    ax.set_ylabel(spec["ylabel"])
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    plot_path = os.path.join("pics", spec["filename"])
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    missing_plot_paths.append(plot_path)
    plt.show()

missing_plot_paths


## Task 2: Robustness to Messy Data with XGBoost

**Hypothesis:** XGBoost can remain competitive on messy data, but unlike TabPFN it needs an explicit preprocessing pipeline for categorical columns. Missing values can be handled by XGBoost internally, while raw categoricals must be encoded before fitting.


## Task 2: Outlier Rate and Outlier Scale

**Hypothesis:** increasing the percentage of numeric outliers should reduce TabPFN performance, and the degradation should be stronger when the outlier scale is larger. This experiment varies outlier rate and outlier scale together while keeping missing values off and categorical columns raw.


In [ ]:
def run_outlier_tabpfn_experiment(
    X_raw,
    y,
    outlier_rates,
    outlier_scales,
    seeds,
    test_size=0.20,
):
    """Evaluate TabPFN across outlier rates and outlier scales."""
    rows = []
    y = np.asarray(y)

    for seed in seeds:
        for outlier_scale in outlier_scales:
            for outlier_rate in outlier_rates:
                X_outlier, metadata = make_messy_tabular_data(
                    X_raw,
                    missing_rate=0.0,
                    outlier_rate=outlier_rate,
                    outlier_scale=outlier_scale,
                    raw_categoricals=True,
                    seed=seed,
                )

                X_train, X_test, y_train, y_test = train_test_split(
                    X_outlier,
                    y,
                    test_size=test_size,
                    random_state=seed,
                    stratify=y,
                )

                model = TabPFNClassifier(random_state=seed)
                model.fit(X_train, y_train)
                y_pred_proba = np.asarray(model.predict_proba(X_test), dtype=float)

                y_pred_proba = np.clip(y_pred_proba, 1e-15, 1.0)
                y_pred_proba = y_pred_proba / y_pred_proba.sum(axis=1, keepdims=True)

                rows.append(
                    {
                        "Seed": seed,
                        "Outlier Rate": outlier_rate,
                        "Outlier Percent": int(outlier_rate * 100),
                        "Outlier Scale": outlier_scale,
                        "ROC AUC": roc_auc_score(y_test, y_pred_proba[:, 1]),
                        "Log Loss": log_loss(y_test, y_pred_proba),
                    }
                )

    return pd.DataFrame(rows)


outlier_rates = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
outlier_scales = [10.0, 20.0]
outlier_eval_seeds = [0, 1, 2]

outlier_tabpfn_results = run_outlier_tabpfn_experiment(
    X,
    y,
    outlier_rates,
    outlier_scales,
    outlier_eval_seeds,
)

outlier_tabpfn_summary = (
    outlier_tabpfn_results
    .groupby(["Outlier Rate", "Outlier Percent", "Outlier Scale"])
    .agg(
        ROC_AUC_mean=("ROC AUC", "mean"),
        ROC_AUC_sem=("ROC AUC", "sem"),
        Log_Loss_mean=("Log Loss", "mean"),
        Log_Loss_sem=("Log Loss", "sem"),
    )
    .reset_index()
    .fillna(0)
)

display(outlier_tabpfn_summary)


In [ ]:
os.makedirs("pics", exist_ok=True)

outlier_plot_specs = [
    {
        "title": "TabPFN ROC AUC vs Outliers",
        "mean_col": "ROC_AUC_mean",
        "sem_col": "ROC_AUC_sem",
        "ylabel": "ROC AUC",
        "filename": "outliers_tabpfn_roc_auc_by_scale.png",
    },
    {
        "title": "TabPFN Log Loss vs Outliers",
        "mean_col": "Log_Loss_mean",
        "sem_col": "Log_Loss_sem",
        "ylabel": "Log Loss",
        "filename": "outliers_tabpfn_log_loss_by_scale.png",
    },
]

outlier_plot_paths = []
colors = {5.0: "tab:blue", 10.0: "tab:orange"}

for spec in outlier_plot_specs:
    fig, ax = plt.subplots(figsize=(7, 5))

    for outlier_scale in outlier_scales:
        scale_summary = outlier_tabpfn_summary[
            outlier_tabpfn_summary["Outlier Scale"] == outlier_scale
        ].sort_values("Outlier Percent")

        ax.errorbar(
            scale_summary["Outlier Percent"],
            scale_summary[spec["mean_col"]],
            yerr=scale_summary[spec["sem_col"]],
            marker="o",
            linewidth=2,
            capsize=4,
            color=colors.get(outlier_scale),
            label=f"Outlier scale {outlier_scale:g}",
        )

    ax.set_title(spec["title"])
    ax.set_xlabel("Outlier Rate (%)")
    ax.set_ylabel(spec["ylabel"])
    ax.legend(title="Outlier Scale")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    plot_path = os.path.join("pics", spec["filename"])
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    outlier_plot_paths.append(plot_path)
    plt.show()

outlier_plot_paths
